In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

INPUT_PATH  = "../../../data/phase2/features_for_model.parquet"
OUTPUT_PATH = "../../../data/phase2/validation_results.parquet"

FEATURE_COLS = [
    "ema_ratio", "rsi_14", "macd_hist", "atr_14",
    "session_quality_enc", "direction_enc", "signal_valid_enc",
]
LABEL_COL = "label"

In [3]:
def train_fold(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
) -> tuple[np.ndarray, StandardScaler, LogisticRegression]:
    """
    Fit StandardScaler + LogisticRegression on training data.
    Return (predicted_probabilities_for_test, fitted_scaler, fitted_model).

    Scaler is fit only on X_train to prevent data leakage.
    LogisticRegression uses C=1.0, max_iter=1000, solver='lbfgs'.
    Returns probabilities for the positive class (label=1).
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    model = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=42)
    model.fit(X_train_scaled, y_train)

    proba = model.predict_proba(X_test_scaled)[:, 1]  # P(label=1)
    return proba, scaler, model


def run_walk_forward(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run walk-forward validation across all folds.

    For each fold N:
      1. Filter train rows (fold==N, split=="train")
      2. Filter test rows  (fold==N, split=="test")
      3. Call train_fold
      4. Append test rows + predicted probability to results

    Returns DataFrame with columns:
      date, s3_key, fold, label, prob_itm, signal_valid_enc, direction_enc
    """
    folds = sorted(df[df["split"] == "test"]["fold"].unique())
    results = []

    for fold_idx in folds:
        train = df[(df["fold"] == fold_idx) & (df["split"] == "train")]
        test  = df[(df["fold"] == fold_idx) & (df["split"] == "test")]

        if len(train) < 10:
            print(f"Fold {fold_idx}: skipping — only {len(train)} train rows")
            continue
        if len(test) == 0:
            print(f"Fold {fold_idx}: skipping — no test rows")
            continue

        X_train = train[FEATURE_COLS]
        y_train = train[LABEL_COL]
        X_test  = test[FEATURE_COLS]

        proba, _, model = train_fold(X_train, y_train, X_test)

        fold_results = test[["date", "s3_key", "fold", LABEL_COL, "signal_valid_enc", "direction_enc"]].copy()
        fold_results["prob_itm"] = proba

        n_pos = int((y_train == 1).sum())
        n_neg = int((y_train == 0).sum())
        print(f"Fold {fold_idx}: train={len(train)} (pos={n_pos}, neg={n_neg}), test={len(test)}, coef={model.coef_[0].round(3)}")
        results.append(fold_results)

    if not results:
        raise RuntimeError("No folds produced results.")
    return pd.concat(results, ignore_index=True)

In [4]:
df = pd.read_parquet(INPUT_PATH)
results = run_walk_forward(df)

print(f"\nTotal test predictions: {len(results)}")
print(f"prob_itm range: {results['prob_itm'].min():.3f} – {results['prob_itm'].max():.3f}")
print(f"Label distribution in test set:\n{results['label'].value_counts()}")

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
results.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

Fold 0: train=444 (pos=180, neg=264), test=71, coef=[-0.563  0.284 -0.137  0.08  -0.158 -1.847  0.395]
Fold 1: train=462 (pos=193, neg=269), test=67, coef=[-0.638  0.181 -0.221 -0.035 -0.519 -1.783  0.615]
Fold 2: train=457 (pos=188, neg=269), test=84, coef=[-0.753  0.105 -0.495  0.103 -0.573 -1.73   0.642]
Fold 3: train=487 (pos=191, neg=296), test=52, coef=[-0.515  0.093 -0.176 -0.066 -0.789 -1.753  0.701]
Fold 4: train=439 (pos=185, neg=254), test=65, coef=[-0.46  -0.046 -0.179 -0.078 -0.774 -1.616  0.784]
Fold 5: train=421 (pos=172, neg=249), test=78, coef=[-0.5    0.169 -0.099 -0.191 -0.89  -1.679  0.884]
Fold 6: train=417 (pos=179, neg=238), test=81, coef=[-1.091  0.004 -0.087 -0.306 -0.619 -1.479  0.439]
Fold 7: train=427 (pos=191, neg=236), test=67, coef=[-0.957  0.17   0.045 -0.166 -0.263 -1.663 -0.019]
Fold 8: train=427 (pos=180, neg=247), test=28, coef=[-0.955  0.132  0.089 -0.3   -0.162 -1.473 -0.135]
Fold 9: train=371 (pos=150, neg=221), test=102, coef=[-1.27   0.245  0.10

Fold 31: train=458 (pos=213, neg=245), test=90, coef=[ 0.303 -0.229  0.197 -0.305 -0.429 -1.915 -0.012]
Fold 32: train=475 (pos=201, neg=274), test=113, coef=[ 0.284 -0.246  0.15  -0.217 -0.599 -1.714  0.122]
Fold 33: train=505 (pos=216, neg=289), test=46, coef=[ 0.21   0.003  0.152 -0.177 -0.523 -1.814  0.144]
Fold 34: train=478 (pos=208, neg=270), test=72, coef=[ 0.298 -0.155 -0.296  0.084 -0.873 -1.855  0.457]
Fold 35: train=455 (pos=180, neg=275), test=82, coef=[ 0.046 -0.145 -0.179  0.043 -0.361 -1.524  0.128]
Fold 36: train=466 (pos=212, neg=254), test=86, coef=[-0.509 -0.092 -0.176  0.089 -0.11  -1.351 -0.045]
Fold 37: train=489 (pos=225, neg=264), test=34, coef=[-1.33  -0.101 -0.238  0.272 -0.419 -1.265  0.379]
Fold 38: train=433 (pos=219, neg=214), test=103, coef=[-1.549  0.124  0.057  0.131 -0.097 -1.432  0.194]
Fold 39: train=423 (pos=191, neg=232), test=66, coef=[-0.723  0.017 -0.086 -0.398  0.032 -1.531  0.131]
Fold 40: train=443 (pos=191, neg=252), test=73, coef=[-0.54   

Fold 50: train=436 (pos=183, neg=253), test=66, coef=[-0.552 -0.432 -1.11  -0.541  0.213 -0.756 -0.383]
Fold 51: train=414 (pos=169, neg=245), test=61, coef=[-0.572 -0.626 -1.076 -0.556  0.131 -0.62  -0.416]
Fold 52: train=412 (pos=166, neg=246), test=23, coef=[-0.677 -0.42  -1.501 -0.558  0.119 -0.894 -0.336]
Fold 53: train=358 (pos=139, neg=219), test=14, coef=[-0.56  -0.341 -1.434 -0.595  0.104 -1.019 -0.332]
Fold 54: train=288 (pos=115, neg=173), test=68, coef=[-0.766 -0.18  -0.927 -1.079  0.097 -0.919 -0.302]
Fold 55: train=319 (pos=121, neg=198), test=78, coef=[-1.142 -0.241 -1.363 -0.725 -0.007 -0.836 -0.238]
Fold 56: train=310 (pos=110, neg=200), test=63, coef=[-0.95   0.079 -1.333 -0.693  0.025 -1.48  -0.144]
Fold 57: train=307 (pos=112, neg=195), test=72, coef=[-0.929 -0.016 -1.476 -0.523  0.11  -1.453 -0.097]
Fold 58: train=318 (pos=112, neg=206), test=75, coef=[-0.911 -0.162 -1.327 -0.75  -0.151 -1.429  0.12 ]
Fold 59: train=370 (pos=139, neg=231), test=51, coef=[-0.885 -0.


Saved to ../../../data/phase2/validation_results.parquet


In [5]:
df = pd.read_parquet(OUTPUT_PATH)
print(df[["date", "s3_key", "fold", "label", "prob_itm"]].head(20).to_string())

                        date  s3_key  fold  label  prob_itm
0  2021-03-11 00:00:00+00:00  EURUSD     0    1.0  0.841059
1  2021-03-12 00:00:00+00:00  EURUSD     0    1.0  0.859150
2  2021-03-15 00:00:00+00:00  EURUSD     0    1.0  0.852528
3  2021-03-16 00:00:00+00:00  EURUSD     0    1.0  0.848750
4  2021-03-17 00:00:00+00:00  EURUSD     0    1.0  0.845264
5  2021-04-28 00:00:00+00:00  EURUSD     0    1.0  0.155726
6  2021-04-29 00:00:00+00:00  EURUSD     0    0.0  0.163161
7  2021-04-30 00:00:00+00:00  EURUSD     0    0.0  0.156368
8  2021-05-03 00:00:00+00:00  EURUSD     0    1.0  0.120379
9  2024-10-30 00:00:00+00:00  USDJPY     0    0.0  0.201989
10 2024-10-31 00:00:00+00:00  USDJPY     0    0.0  0.196096
11 2024-12-30 00:00:00+00:00  USDJPY     0    0.0  0.186638
12 2021-09-16 00:00:00+00:00      CL     0    0.0  0.112634
13 2021-09-17 00:00:00+00:00      CL     0    0.0  0.101107
14 2021-09-20 00:00:00+00:00      CL     0    0.0  0.088118
15 2021-09-21 00:00:00+00:00      CL    